# Payment gateway API testing lab

An interactive workbench on top of the running service: request waves, the state machine,
idempotency, latency. Reusable code lives in `gateway_client.py` next to this notebook.

**Setup:**
1. `docker compose up -d` — the service at `http://localhost:8000`
2. `uv sync --group notebooks` and pick the Python kernel from `.venv`
3. `PROVIDER_MODE` in the cell below must match the service's `FAKE_PROVIDER_MODE`
   (`.env`, default `manual`)

A top-to-bottom run (Run All) passes end-to-end in both manual and auto mode:
cells with asserts fail if the API behaves unexpectedly.

In [11]:
import uuid

import pandas as pd
from gateway_client import (
    GatewayClient,
    GatewayConfig,
    run_lifecycle,
    run_wave,
    scenario_for,
    summarize_lifecycles,
    summarize_waves,
    wait_for_status,
)
from IPython.display import display

pd.set_option("display.max_colwidth", 120)

# Must match the service's FAKE_PROVIDER_MODE (.env, default manual):
# manual — the notebook sends the callbacks, auto — the service sends them
# to itself and the notebook only polls the statuses.
PROVIDER_MODE = "auto"

# Defaults match .env.example; for a different setup override the fields:
# GatewayConfig(base_url="http://localhost:8001", api_key="...").
config = GatewayConfig()
client = GatewayClient(config)

health = await client.health()
if health.status_code != 200:
    raise RuntimeError(
        f"Service unavailable at {config.base_url}: {health.error or health.data}. "
        "Start it: docker compose up -d"
    )
health.data

{'status': 'ok', 'db': 1}

## 1. Smoke: one payment through the full cycle

Create a payment with the `success` scenario and drive it to a terminal status.

In [12]:
smoke = await run_lifecycle(client, mode=PROVIDER_MODE, scenario="success")

print("payment_id:", smoke.payment_id)
print("trajectory:", " -> ".join(smoke.trajectory))
print("final status:", smoke.final_status)
assert smoke.final_status == "success", smoke

payment_id: 019f4717-196f-7f13-9fd0-0cbe08346223
trajectory: pending -> processing -> success
final status: success


## 2. Full lifecycle as a wave

N payments concurrently, six scenarios laid out round-robin. Each payment
must reach its expected terminal status. The table below shows how many
payments of each scenario ended up in which terminal status.

In manual mode a callback answered with 409 is retried (like webhook redelivery
at a real PSP): the service responds before it commits the transaction, so the
next callback may still see the old status. The count of such retries is printed
below the table — it measures the commit visibility lag.

In [13]:
LIFECYCLE_N = 300
scenarios = [scenario_for(i) for i in range(LIFECYCLE_N)]

lifecycle_report = await run_wave(
    LIFECYCLE_N,
    lambda i: run_lifecycle(client, mode=PROVIDER_MODE, scenario=scenarios[i]),
    label=f"lifecycle x{LIFECYCLE_N}",
)
lifecycles = summarize_lifecycles(lifecycle_report.results)

print(f"wall-time: {lifecycle_report.wall_time_s:.2f}s")
total_retries = int(lifecycles["retries"].sum())
if total_retries:
    print(f"callback retries due to commit visibility lag: {total_retries}")
lifecycles.groupby(["scenario", "final_status"], dropna=False).size().unstack(fill_value=0)

wall-time: 6.84s


final_status,error,failed,success
scenario,,,
error,50,0,0
fraud,0,50,0
insufficient_funds,0,50,0
limit_exceeded,0,50,0
success,0,0,50
timeout,0,50,0


In [14]:
not_ok = lifecycles[~lifecycles["ok"]]
assert not_ok.empty, not_ok.to_string()
print(f"all {len(lifecycles)} payments reached their expected terminal statuses")

all 300 payments reached their expected terminal statuses


## 3. State machine: allowed and forbidden transitions (manual)

```
created -> pending -> processing -> success
              |           |-------> error
              |           \\------> failed (insufficient_funds | fraud | limit_exceeded)
              \\-> failed (timeout)
```

We check response codes: forbidden transitions (409), invalid failure reasons
(422), nonexistent payments (404), wrong keys (401). In auto mode this section
is skipped: the service's callbacks would race the notebook's.

In [ ]:
if PROVIDER_MODE != "manual":
    print("skipped: in auto mode the service's callbacks would race the notebook's")
else:
    victim = await client.create_payment()
    pid = victim.payment_id
    missing_id = str(uuid.uuid4())
    cases = []

    async def check(name, expected_code, request_coro):
        result = await request_coro
        cases.append(
            {
                "case": name,
                "expected": expected_code,
                "actual": result.status_code,
                "ok": result.status_code == expected_code,
            }
        )

    # The service responds before committing the transaction, so after each
    # successful transition we wait until the new status becomes visible —
    # otherwise the next check may still see the old state.
    assert await wait_for_status(client, pid, "pending"), "payment not visible in pending"

    await check("pending -> success forbidden", 409, client.send_callback(pid, "success"))
    await check(
        "failed(fraud) from pending forbidden",
        422,
        client.send_callback(pid, "failed", failure_reason="fraud"),
    )
    await check("pending -> processing allowed", 200, client.send_callback(pid, "processing"))
    assert await wait_for_status(client, pid, "processing"), "payment not visible in processing"
    await check(
        "failed(timeout) from processing forbidden",
        422,
        client.send_callback(pid, "failed", failure_reason="timeout"),
    )
    await check("processing -> success allowed", 200, client.send_callback(pid, "success"))
    assert await wait_for_status(client, pid, "success"), "payment not visible in success"
    await check(
        "success is terminal: -> processing forbidden",
        409,
        client.send_callback(pid, "processing"),
    )
    await check("failed without failure_reason", 422, client.send_callback(pid, "failed"))
    await check(
        "callback with unknown payment_id",
        404,
        client.send_callback(missing_id, "success"),
    )
    await check("GET for a nonexistent payment", 404, client.get_payment(missing_id))
    await check("unknown provider_id", 422, client.create_payment(provider_id=999))

    stranger = GatewayClient(GatewayConfig(api_key="wrong-key", callback_secret="wrong"))
    await check("wrong X-API-Key", 401, stranger.create_payment())
    await check("wrong X-Callback-Secret", 401, stranger.send_callback(pid, "success"))
    await stranger.aclose()

    machine_checks = pd.DataFrame(cases)
    display(machine_checks)
    assert machine_checks["ok"].all(), "mismatches with expected response codes"

## 4. Creation idempotency (the Stripe model)

One `Idempotency-Key` is sent N times concurrently: exactly one payment is created,
the remaining responses are replays with the same `payment_id` and the
`Idempotency-Replayed: true` header. The same key with a different body — `422`.

In [5]:
IDEMPOTENCY_N = 20
idempotency_key = f"nb-{uuid.uuid4()}"

idempotency_report = await run_wave(
    IDEMPOTENCY_N,
    lambda i: client.create_payment(amount="42.00", idempotency_key=idempotency_key),
    label=f"idempotent create x{IDEMPOTENCY_N}",
)
created_results = [r for r in idempotency_report.results if r.status_code == 201]
unique_ids = {r.payment_id for r in created_results}
replayed = [r for r in created_results if r.headers.get("idempotency-replayed") == "true"]

print(f"201 responses: {len(created_results)} of {IDEMPOTENCY_N}")
print(f"unique payment_ids: {len(unique_ids)}")
print(f"replays (Idempotency-Replayed header): {len(replayed)}")
assert len(created_results) == IDEMPOTENCY_N
assert len(unique_ids) == 1, unique_ids
assert len(replayed) == IDEMPOTENCY_N - 1

ответов 201: 20 из 20
уникальных payment_id: 1
реплеев (заголовок Idempotency-Replayed): 19


In [6]:
mismatch = await client.create_payment(amount="43.00", idempotency_key=idempotency_key)

print(mismatch.status_code, mismatch.data)
assert mismatch.status_code == 422

422 {'detail': "Idempotency key 'nb-6c27c30d-8bb9-436c-81c4-69c86efd357a' was already used with a different request body"}


## 5. Creation waves: 1 / 10 / 100 / 1000

Each wave is n concurrent `POST /payments` without idempotency keys.
Summary: HTTP code distribution, p50/p95/p99 latency, RPS.

In manual mode created payments stay in `pending`; in auto mode the service
additionally sends itself two callbacks per payment — extra load. The tail of
those callbacks keeps hitting the service for a while after the wave, so the
load section goes last: it does not interfere with the functional checks above.

In [10]:
WAVE_SIZES = [1, 10, 100, 1000]

wave_reports = []
for size in WAVE_SIZES:
    report = await run_wave(size, lambda i: client.create_payment(), label=f"create x{size}")
    wave_reports.append(report)

summarize_waves(wave_reports)

,wave,requests,network_errors,http_codes,p50_ms,p95_ms,p99_ms,wall_s,rps
0,create x1,1,0,201:1,15.1,15.1,15.1,0.02,65.8
1,create x10,10,0,201:10,52.6,60.1,60.2,0.07,152.9
2,create x100,100,0,201:100,218.4,359.3,374.4,0.40,249.7
3,create x1000,1000,0,201:1000,4568.3,7720.2,7991.7,8.33,120.1


## 6. How to extend

- **A new endpoint** — a method on `GatewayClient` modeled after `create_payment`:
  build the headers/body and return `self._request(...)`. Plus a scenario cell here.
- **A new provider scenario** — an entry in `SCENARIO_CALLBACKS` and
  `SCENARIO_EXPECTED_FINAL` in `gateway_client.py`.
- **A new load profile** — `run_wave(n, factory, concurrency=...)`:
  factory receives the request index, so any calls can be mixed within one wave.
- **New state machine checks** — one more `await check(...)` in section 3;
  after each successful transition remember the `wait_for_status` barrier.

In [ ]:
await client.aclose()
print("client closed")